# EmpowerLens — Cascade runner (single backbone, disk-safe)

**Why this version is different from the one that failed:** the previous run trained
2 backbones × 3 seeds × 3 tasks = 18 configs. Each left ~2-2.5GB of checkpoint data
behind (the Trainer's internal `checkpoint-XXX/` folder with optimizer state, on top
of the final saved model) and nothing was copied to `/kaggle/working/` until the very
last cell — so when disk filled up partway through, the whole run was lost.

This version fixes both problems:
1. **One backbone only — `mental/mental-roberta-base`.** It beat DeBERTa-v3-base on
   every metric in the prior side-by-side run (binary positive_class_f1 0.821 vs 0.807,
   multiclass macro_f1_10 0.188 vs 0.172, multilabel macro_f1 0.279 vs 0.193) and had a
   much smaller val→test overfitting gap. Cuts total runs from 18 to 9.
2. **Disk-safe by construction:** after every single train+evaluate pair, the wasteful
   `checkpoint-XXX/` subfolder (model + optimizer + scheduler) is deleted immediately,
   keeping only the final saved model that `evaluate.py`/`evaluate_cascade.py` actually
   read. Results are synced to `/kaggle/working/` after every stage, not just at the end,
   so a failure partway through only costs you that stage, not the whole session.

**Before running:**
1. Settings -> **Accelerator: GPU**, **Internet: On**.
2. `data/splits_combined/` already committed and pushed on the branch below.
3. `src/losses.py`, `src/make_splits_cascade.py`, `src/evaluate_cascade.py`, and the
   updated `src/train_transformer.py` pushed to the branch too.
4. `HF_TOKEN` Kaggle secret set (mental-roberta-base is gated on the Hub).

**Pipeline, in order:** binary (Stage 1, 3 seeds) -> multiclass (comparison track,
3 seeds) -> multilabel Stage 2 distorted-only (3 seeds) -> cascade eval (3 seed pairs)
-> comparison tables. Every stage prints its results as soon as it's produced and syncs
to `/kaggle/working/` immediately after.

In [ ]:
# 1. Clone the repo and install the transformer stack.
REPO_URL = "https://github.com/lumia-Qcode/EmpowerLens.git"
BRANCH   = "lumia-space"

!rm -rf empowerlens && git clone --branch $BRANCH $REPO_URL empowerlens
%cd empowerlens
!pip install -q -r requirements-transformer.txt
!pip install -q sentencepiece protobuf

In [ ]:
# 1a. mental/mental-roberta-base is GATED on the Hub — accept its terms at
#     https://huggingface.co/mental/mental-roberta-base while logged in, then add a
#     Kaggle Secret named HF_TOKEN (Add-ons -> Secrets) with a read-scope HF token.
from huggingface_hub import login
try:
    from kaggle_secrets import UserSecretsClient
    hf_token = UserSecretsClient().get_secret("HF_TOKEN")
    login(token=hf_token)
    print("Logged in to Hugging Face Hub.")
except Exception as e:
    print(f"[warn] no working HF_TOKEN secret ({e}) — training will 401 until this is set.")

In [ ]:
# 1b. Baseline disk check — compare against this after each stage below to catch a
#     runaway before it becomes a failed session.
!df -h /kaggle/working

In [ ]:
# 1c. Shared helpers used by every stage below:
#     - run_and_report(...): trains one config, evaluates it, deletes the wasteful
#       checkpoint-XXX/ optimizer-state subfolder, and prints the test metrics
#       immediately so you see results as each seed finishes, not just at the end.
#     - sync(folder): copies a results/ dir to /kaggle/working/ right away.
import json
import subprocess
from pathlib import Path

MODEL = "mental/mental-roberta-base"
TAG = MODEL.split("/")[-1]
SEEDS = (42, 1337, 2024)

def sh(cmd):
    print(f"$ {cmd}")
    r = subprocess.run(cmd, shell=True)
    if r.returncode != 0:
        print(f"[FAILED] exit code {r.returncode}: {cmd}")
    return r.returncode

def sync(folder):
    Path(f"/kaggle/working/{folder}").mkdir(parents=True, exist_ok=True)
    sh(f"cp -r {folder}/* /kaggle/working/{folder}/")
    print(f"[synced] {folder} -> /kaggle/working/{folder}")

def show_test_metrics(out_dir, task, seed):
    p = Path(out_dir) / f"eval_{TAG}_{task}_{seed}.json"
    if not p.exists():
        print(f"  [seed {seed}] eval JSON not found at {p} — evaluate step likely failed above")
        return
    m = json.loads(p.read_text())["splits"]["test"]["metrics"]
    if task == "binary":
        print(f"  [seed {seed}] test weighted_f1={m['weighted_f1']:.3f}  positive_class_f1={m['positive_class_f1']:.3f}")
    elif task == "multiclass":
        print(f"  [seed {seed}] test weighted_f1={m['weighted_f1']:.3f}  macro_f1_10={m['macro_f1_10']:.3f}")
    else:
        print(f"  [seed {seed}] test weighted_f1={m['weighted_f1']:.3f}  macro_f1={m['macro_f1']:.3f}")

def run_and_report(task, splits_dir, out_dir, seed, extra_flags=""):
    ckpt = f"checkpoints/{task}_{TAG}_{seed}"
    rc = sh(
        f"python -m src.train_transformer --task {task} --model {MODEL} --seed {seed} "
        f"--device auto --splits {splits_dir} {extra_flags}"
    )
    if rc == 0:
        rc = sh(f"python -m src.evaluate --checkpoint {ckpt} --reference --splits {splits_dir} --out {out_dir}")
    # Delete the Trainer's internal checkpoint-XXX/ (model+optimizer+scheduler) regardless
    # of success/failure above — it's never read by evaluate.py, only the ckpt root is.
    sh(f"rm -rf {ckpt}/checkpoint-*")
    show_test_metrics(out_dir, task, seed)
    return ckpt

In [ ]:
# Step 1 — derive Stage 2's distorted-only splits from data/splits_combined.
COMBINED_SPLITS = "data/splits_combined"
STAGE2_SPLITS   = "data/splits_stage2"

!python -m src.make_splits_cascade --source $COMBINED_SPLITS --out $STAGE2_SPLITS --force

## Step 2 — Stage 1: binary model, 3 seeds

Trained on the FULL `data/splits_combined`. Results print inline as each seed finishes; synced to `/kaggle/working/results_stage1/` right after.

In [ ]:
STAGE1_OUT = "results_stage1"
!mkdir -p $STAGE1_OUT

print(f"=== Stage 1 binary: {MODEL} ===")
for seed in SEEDS:
    run_and_report("binary", COMBINED_SPLITS, STAGE1_OUT, seed)

sync(STAGE1_OUT)
!df -h /kaggle/working

## Step 2b — Multiclass (11-class), 3 seeds — comparison track, not part of the cascade

Same data as Step 2. Benefits from two fixes not present in the earlier `results_combined` run: `metric_for_best_model` now selects on `macro_f1_10` instead of `macro_f1`, and `--label-smoothing 0.1` hedges against the CODIPAS/Annotated label-definition mismatch. Written to `results_multiclass_v2/` so the old numbers aren't overwritten.

In [ ]:
MULTICLASS_OUT = "results_multiclass_v2"
!mkdir -p $MULTICLASS_OUT

print(f"=== Multiclass (11-class): {MODEL} ===")
for seed in SEEDS:
    run_and_report(
        "multiclass", COMBINED_SPLITS, MULTICLASS_OUT, seed,
        extra_flags="--label-smoothing 0.1 --lr-scheduler cosine --early-stopping-patience 2",
    )

sync(MULTICLASS_OUT)
!df -h /kaggle/working

## Step 3 — Stage 2: multilabel head, distorted-only, 3 seeds

Trained on `data/splits_stage2` — never sees `no_distortion`, so `macro_f1` here is the honest "can it tell the 10 types apart" number. Uses focal loss + layer-wise LR decay, the two new levers for the minority classes (`all_or_nothing`, `mental_filter`, `personalization`).

In [ ]:
STAGE2_OUT = "results_stage2"
!mkdir -p $STAGE2_OUT

print(f"=== Stage 2 multilabel: {MODEL} (focal + LLRD) ===")
for seed in SEEDS:
    run_and_report(
        "multilabel", STAGE2_SPLITS, STAGE2_OUT, seed,
        extra_flags=(
            "--loss focal --focal-gamma 2.0 --llrd --llrd-decay 0.9 --lr 3e-5 "
            "--lr-scheduler cosine --grad-accum 2 --early-stopping-patience 2"
        ),
    )

sync(STAGE2_OUT)
!df -h /kaggle/working

## Step 4 — end-to-end cascade evaluation (the number that counts)

`results_stage2/` above scores Stage 2 in isolation on distorted-only inputs — that looks better than reality since it never sees Stage 1's false negatives. This chains Stage 1 -> Stage 2 and scores the composed prediction against the FULL val/test set.

In [ ]:
CASCADE_OUT = "results_cascade"
!mkdir -p $CASCADE_OUT

print(f"=== Cascade eval: {TAG} (Stage 1 + Stage 2, matched seeds) ===")
for seed in SEEDS:
    stage1_ckpt = f"checkpoints/binary_{TAG}_{seed}"
    stage2_ckpt = f"checkpoints/multilabel_{TAG}_{seed}"
    sh(
        f"python -m src.evaluate_cascade --stage1-checkpoint {stage1_ckpt} "
        f"--stage2-checkpoint {stage2_ckpt} --splits {COMBINED_SPLITS} --out {CASCADE_OUT}"
    )

sync(CASCADE_OUT)
!df -h /kaggle/working

## Step 5 — compare: multilabel flat vs cascade, multiclass old vs fixed

In [ ]:
import pandas as pd

SOURCES = {
    "results_combined (flat, old)": "results_combined",
    "results_cascade (composed)": CASCADE_OUT,
    "results_multiclass_v2 (fixed)": MULTICLASS_OUT,
}

frames = []
for label, folder in SOURCES.items():
    p = Path(folder) / "paper_comparison.csv"
    if p.exists():
        d = pd.read_csv(p)
        d["results_dir"] = label
        frames.append(d)
    else:
        print(f"[skip] {p} not found")

all_results = pd.concat(frames, ignore_index=True)

view_ml = all_results[(all_results["task"] == "multilabel") & (all_results["split"] == "test")]
print("=== multilabel: flat vs cascade (test) ===")
print(view_ml.groupby(["results_dir", "model"])[["weighted_f1", "macro_f1"]].agg(["mean", "std"]).round(3))

view_mc = all_results[(all_results["task"] == "multiclass") & (all_results["split"] == "test")]
print("\n=== multiclass: old vs fixed (test) ===")
print(view_mc.groupby(["results_dir", "model"])[["weighted_f1", "macro_f1_10"]].agg(["mean", "std"]).round(3))

all_results.to_csv(f"{CASCADE_OUT}/flat_vs_cascade_vs_multiclass_comparison.csv", index=False)
sync(CASCADE_OUT)
print(f"\nWrote and synced flat_vs_cascade_vs_multiclass_comparison.csv")

In [ ]:
# Final safety net — everything above already synced incrementally after each stage,
# this just re-confirms all four folders are present in /kaggle/working/.
for folder in (STAGE1_OUT, MULTICLASS_OUT, STAGE2_OUT, CASCADE_OUT):
    sync(folder)
!ls -la /kaggle/working
!df -h /kaggle/working